# Module 6.1: 高级优化技术 (Advanced Optimization)

## 1. 本章概览 (Overview)

### 📚 学习目标

1. **自适应优化器**：理解 Adam、AdamW 等现代优化器
2. **学习率调度**：掌握 warmup、cosine annealing 等策略
3. **梯度累积**：用有限内存模拟大批次训练
4. **混合精度训练**：使用 FP16/BF16 加速训练
5. **训练稳定性**：梯度裁剪、层归一化等技巧
6. **综合流程**：构建完整的高效训练管道

### 🎯 核心问题

- 为什么 Adam 比 SGD 更适合 Transformer？
- 如何选择学习率调度策略？
- 梯度累积如何节省内存？
- 混合精度训练为什么更快？
- 如何防止训练不稳定？


### 🏢 业务场景

本章技术将应用于 `电商客服智能助理` 场景：
- 客服模型训练 loss 震荡不收敛？→ 优化器选择与学习率调度策略
- 训练中 loss 突然飙升怎么办？→ 梯度裁剪与warmup 的稳定化技巧

### 🔑 核心概念速览

**梯度累积 (Gradient Accumulation)** 是一种通过多次前向-反向传播累加梯度后再统一更新参数的技术，从而用有限显存模拟大批次训练。在本章场景中，它用于突破 GPU 内存限制，实现等效大 batch size 的训练效果。

**混合精度训练 (Mixed Precision Training)** 是在训练过程中同时使用 FP32 和 FP16/BF16 两种浮点精度的技术，在保持模型精度的前提下显著加速计算并降低显存占用。在本章场景中，它用于实现 2-3 倍的训练加速。

**梯度裁剪 (Gradient Clipping)** 是在反向传播后对梯度的值或范数进行截断的技术，防止梯度过大导致参数更新不稳定。在本章场景中，它用于解决深层网络（如 Transformer）中常见的梯度爆炸问题。

### 🗺️ 优化技术地图

```
基础优化器 (SGD, Momentum)
    ↓
自适应优化器 (Adam, AdamW)
    ↓
学习率调度 (Warmup, Cosine)
    ↓
内存优化 (Gradient Accumulation)
    ↓
速度优化 (Mixed Precision)
    ↓
稳定性技巧 (Gradient Clipping)
    ↓
完整训练流程
```

### ⚡ 为什么需要高级优化？

**大模型训练的挑战**：
- **内存限制**：模型太大，批次太小
- **训练时间**：需要数周甚至数月
- **不稳定性**：梯度爆炸、损失发散
- **收敛速度**：需要快速找到最优解

**解决方案**：
- 自适应学习率 → 更快收敛
- 学习率调度 → 更好的最终性能
- 梯度累积 → 模拟大批次
- 混合精度 → 2-3倍加速
- 梯度裁剪 → 训练稳定

### ⏱️ 预计学习时间：3-4小时

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
import matplotlib.pyplot as plt
import time

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)

# Configure matplotlib
plt.rcParams['figure.figsize'] = (12, 6)

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Using device: {device}")
print(f"✓ PyTorch version: {torch.__version__}")
print("✓ Libraries imported")

## 2. 优化器基础 (Optimizer Fundamentals)

### 2.1 梯度下降回顾

**基本思想**：沿着损失函数的负梯度方向更新参数

$$\theta_{t+1} = \theta_t - \eta \nabla L(\theta_t)$$

其中：
- $\theta$：模型参数
- $\eta$：学习率
- $\nabla L$：损失函数的梯度

**三种变体**：

1. **批量梯度下降（Batch GD）**
   - 使用全部数据计算梯度
   - 稳定但慢

2. **随机梯度下降（SGD）**
   - 每次使用一个样本
   - 快但不稳定

3. **小批量梯度下降（Mini-batch GD）**
   - 使用小批次数据
   - 平衡速度和稳定性

### 2.2 动量（Momentum）

**问题**：SGD 在峡谷地形中震荡，收敛慢

**解决方案**：累积历史梯度，加速收敛

$$v_{t+1} = \beta v_t + \nabla L(\theta_t)$$
$$\theta_{t+1} = \theta_t - \eta v_{t+1}$$

**效果**：
- 加速相关方向
- 抑制震荡
- 更快收敛

**典型值**：$\beta = 0.9$

### 2.3 为什么需要自适应优化器？

**SGD + Momentum 的局限**：
- 所有参数使用相同学习率
- 需要仔细调整学习率
- 对稀疏梯度不友好

**自适应优化器的优势**：
- 每个参数独立的学习率
- 自动调整学习率
- 更鲁棒，更容易使用

In [ ]:
# 🔬 Micro Practice: Compare SGD and SGD+Momentum
# Goal: Visualize convergence paths of different optimizers
# Expected outcome: See how momentum accelerates convergence

# Define a simple 2D optimization problem (Rosenbrock function)
def rosenbrock(x, y):
    """Rosenbrock function: f(x,y) = (1-x)^2 + 100(y-x^2)^2"""
    return (1 - x)**2 + 100 * (y - x**2)**2

def rosenbrock_grad(x, y):
    """Gradient of Rosenbrock function"""
    dx = -2 * (1 - x) - 400 * x * (y - x**2)
    dy = 200 * (y - x**2)
    return dx, dy

# Optimize with SGD
def optimize_sgd(start_x, start_y, lr=0.001, steps=1000):
    """Optimize using SGD"""
    x, y = start_x, start_y
    path = [(x, y)]
    
    for _ in range(steps):
        dx, dy = rosenbrock_grad(x, y)
        x -= lr * dx
        y -= lr * dy
        path.append((x, y))
    
    return np.array(path)

# Optimize with SGD + Momentum
def optimize_momentum(start_x, start_y, lr=0.001, beta=0.9, steps=1000):
    """Optimize using SGD with momentum"""
    x, y = start_x, start_y
    vx, vy = 0, 0
    path = [(x, y)]
    
    for _ in range(steps):
        dx, dy = rosenbrock_grad(x, y)
        vx = beta * vx + dx
        vy = beta * vy + dy
        x -= lr * vx
        y -= lr * vy
        path.append((x, y))
    
    return np.array(path)

# Run optimization
start_x, start_y = -1.0, 1.0
sgd_path = optimize_sgd(start_x, start_y, lr=0.001, steps=500)
momentum_path = optimize_momentum(start_x, start_y, lr=0.001, beta=0.9, steps=500)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Create contour plot
x_range = np.linspace(-1.5, 1.5, 100)
y_range = np.linspace(-0.5, 2.5, 100)
X, Y = np.meshgrid(x_range, y_range)
Z = rosenbrock(X, Y)

for ax, path, title in zip(axes, [sgd_path, momentum_path], 
                            ['SGD', 'SGD + Momentum']):
    ax.contour(X, Y, Z, levels=np.logspace(-1, 3, 20), cmap='viridis', alpha=0.6)
    ax.plot(path[:, 0], path[:, 1], 'r.-', linewidth=2, markersize=3, alpha=0.7)
    ax.plot(start_x, start_y, 'go', markersize=10, label='Start')
    ax.plot(1, 1, 'r*', markersize=15, label='Optimum')
    ax.set_xlabel('x', fontsize=12)
    ax.set_ylabel('y', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("优化结果对比：")
print(f"SGD 最终位置: ({sgd_path[-1, 0]:.4f}, {sgd_path[-1, 1]:.4f})")
print(f"Momentum 最终位置: ({momentum_path[-1, 0]:.4f}, {momentum_path[-1, 1]:.4f})")
print(f"最优解: (1.0000, 1.0000)")
print("\n观察：Momentum 收敛更快，路径更平滑")

## 3. 自适应优化器 (Adaptive Optimizers)

### 3.1 Adam 优化器

**Adam (Adaptive Moment Estimation)** 是最流行的深度学习优化器。

**核心思想**：结合动量和自适应学习率

**算法步骤**：

1. **计算一阶矩估计（动量）**：
   $$m_t = \beta_1 m_{t-1} + (1-\beta_1)\nabla L(\theta_t)$$

2. **计算二阶矩估计（梯度平方的移动平均）**：
   $$v_t = \beta_2 v_{t-1} + (1-\beta_2)(\nabla L(\theta_t))^2$$

3. **偏差修正**：
   $$\hat{m}_t = \frac{m_t}{1-\beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1-\beta_2^t}$$

4. **参数更新**：
   $$\theta_{t+1} = \theta_t - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

**超参数**：
- $\eta$：学习率（通常 0.001）
- $\beta_1$：一阶矩衰减率（通常 0.9）
- $\beta_2$：二阶矩衰减率（通常 0.999）
- $\epsilon$：数值稳定性（通常 1e-8）

**优势**：
- 每个参数自适应学习率
- 对稀疏梯度友好
- 对超参数不敏感
- 收敛快

**为什么 Adam 适合 Transformer？**
- Transformer 参数多，维度高
- 不同层的梯度尺度差异大
- 需要快速收敛

In [ ]:
# 🔬 Micro Practice: Implement Adam from Scratch
# Goal: Build Adam optimizer and understand its mechanics
# Expected outcome: Working Adam implementation

class AdamOptimizer:
    """Adam optimizer from scratch"""
    
    def __init__(self, params, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params = list(params)
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        
        # Initialize moment estimates
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]
        self.t = 0
    
    def step(self):
        """Perform one optimization step"""
        self.t += 1
        
        for i, param in enumerate(self.params):
            if param.grad is None:
                continue
            
            grad = param.grad.data
            
            # Update biased first moment estimate
            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * grad
            
            # Update biased second moment estimate
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * grad**2
            
            # Bias correction
            m_hat = self.m[i] / (1 - self.beta1**self.t)
            v_hat = self.v[i] / (1 - self.beta2**self.t)
            
            # Update parameters
            param.data -= self.lr * m_hat / (torch.sqrt(v_hat) + self.eps)
    
    def zero_grad(self):
        """Zero out gradients"""
        for param in self.params:
            if param.grad is not None:
                param.grad.zero_()


# Test on a simple problem
print("=" * 60)
print("测试 Adam 优化器")
print("=" * 60)

# Create a simple model
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 1)
    
    def forward(self, x):
        return self.fc(x)

# Generate toy data
torch.manual_seed(42)
X = torch.randn(100, 10)
y = torch.randn(100, 1)

# Train with custom Adam
model = SimpleModel()
optimizer = AdamOptimizer(model.parameters(), lr=0.01)
criterion = nn.MSELoss()

losses = []
for epoch in range(100):
    optimizer.zero_grad()
    output = model(X)
    loss = criterion(output, y)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

print(f"\n初始损失: {losses[0]:.4f}")
print(f"最终损失: {losses[-1]:.4f}")
print(f"损失下降: {(losses[0] - losses[-1]) / losses[0] * 100:.1f}%")

# Compare with PyTorch's Adam
model2 = SimpleModel()
optimizer2 = optim.Adam(model2.parameters(), lr=0.01)

losses2 = []
for epoch in range(100):
    optimizer2.zero_grad()
    output = model2(X)
    loss = criterion(output, y)
    loss.backward()
    optimizer2.step()
    losses2.append(loss.item())

print(f"\nPyTorch Adam 最终损失: {losses2[-1]:.4f}")
print(f"差异: {abs(losses[-1] - losses2[-1]):.6f}")
print("\n✓ Adam 实现验证成功！")

### 3.2 AdamW 和其他变体

**AdamW (Adam with Decoupled Weight Decay)**

**问题**：Adam 中的 L2 正则化与自适应学习率交互，效果不佳

**解决方案**：将权重衰减从梯度中解耦

$$\theta_{t+1} = \theta_t - \eta \left(\frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon} + \lambda \theta_t\right)$$

**优势**：
- 更好的泛化性能
- 正则化效果更稳定
- Transformer 训练的标准选择

**其他自适应优化器**：

1. **AdaGrad**
   - 累积所有历史梯度平方
   - 学习率单调递减
   - 适合稀疏数据

2. **RMSprop**
   - 使用移动平均而非累积
   - 解决 AdaGrad 学习率过快衰减的问题
   - Adam 的前身

3. **AdaMax**
   - 使用无穷范数代替 L2 范数
   - 对大梯度更鲁棒

**优化器选择指南**：

| 任务类型 | 推荐优化器 | 原因 |
|---------|-----------|------|
| **Transformer** | AdamW | 最佳泛化性能 |
| **CNN** | SGD + Momentum | 更好的最终性能 |
| **RNN** | Adam | 处理梯度消失 |
| **稀疏数据** | AdaGrad | 自适应稀疏梯度 |
| **强化学习** | Adam | 非平稳目标 |

In [ ]:
# Compare different optimizers
print("=" * 60)
print("对比不同优化器")
print("=" * 60)

# Create a more complex model
class MLPModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(20, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    
    def forward(self, x):
        return self.layers(x)

# Generate data
torch.manual_seed(42)
X_train = torch.randn(500, 20)
y_train = torch.randn(500, 1)

# Test different optimizers
optimizers_config = {
    'SGD': {'class': optim.SGD, 'params': {'lr': 0.01}},
    'SGD+Momentum': {'class': optim.SGD, 'params': {'lr': 0.01, 'momentum': 0.9}},
    'Adam': {'class': optim.Adam, 'params': {'lr': 0.001}},
    'AdamW': {'class': optim.AdamW, 'params': {'lr': 0.001, 'weight_decay': 0.01}},
    'RMSprop': {'class': optim.RMSprop, 'params': {'lr': 0.001}}
}

results = {}
num_epochs = 100

for name, config in optimizers_config.items():
    model = MLPModel()
    optimizer = config['class'](model.parameters(), **config['params'])
    criterion = nn.MSELoss()
    
    losses = []
    for epoch in range(num_epochs):
        optimizer.zero_grad()
        output = model(X_train)
        loss = criterion(output, y_train)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    
    results[name] = losses
    print(f"{name:15s} - 最终损失: {losses[-1]:.6f}")

# Visualize comparison
plt.figure(figsize=(12, 6))
for name, losses in results.items():
    plt.plot(losses, label=name, linewidth=2)

plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('优化器对比', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.tight_layout()
plt.show()

print("\n观察：")
print("- Adam 和 AdamW 收敛最快")
print("- SGD 需要更多 epoch")
print("- Momentum 显著改善 SGD")
print("- AdamW 通常有更好的泛化性能")
print("\n✓ 优化器对比完成！")

## 4. 学习率调度 (Learning Rate Scheduling)

### 4.1 为什么需要学习率调度？

**固定学习率的问题**：
- 太大：训练不稳定，可能发散
- 太小：收敛太慢
- 难以找到最优值

**调度的好处**：
- 开始时大学习率 → 快速接近最优解
- 后期小学习率 → 精细调整，找到更好的最小值
- 更好的泛化性能

### 4.2 常见调度策略

**1. 线性衰减（Linear Decay）**

$$\eta_t = \eta_0 \left(1 - \frac{t}{T}\right)$$

- 简单直接
- 线性递减到 0

**2. 指数衰减（Exponential Decay）**

$$\eta_t = \eta_0 \cdot \gamma^t$$

- 快速衰减
- $\gamma$ 通常为 0.95-0.99

**3. 余弦退火（Cosine Annealing）**

$$\eta_t = \eta_{\min} + \frac{1}{2}(\eta_{\max} - \eta_{\min})\left(1 + \cos\left(\frac{t}{T}\pi\right)\right)$$

- 平滑衰减
- 可以带重启（Cosine Annealing with Warm Restarts）

**4. Warmup + Decay**

$$\eta_t = \begin{cases} 
\eta_{\max} \frac{t}{T_{\text{warmup}}} & t < T_{\text{warmup}} \\
\eta_{\max} \cdot \text{decay}(t - T_{\text{warmup}}) & t \geq T_{\text{warmup}}
\end{cases}$$

- Transformer 训练的标准做法
- Warmup 防止早期不稳定

### 4.3 Warmup 的重要性

**为什么大模型需要 Warmup？**

1. **参数初始化**：随机初始化的参数可能导致大梯度
2. **Adam 的偏差**：早期的矩估计不准确
3. **批归一化**：统计量需要时间稳定
4. **防止发散**：大学习率可能导致早期损失爆炸

**Warmup 步数选择**：
- 小模型：1000-2000 步
- 大模型（BERT, GPT）：10000-40000 步
- 经验法则：总步数的 1-10%

In [ ]:
# 🔬 Micro Practice: Implement Warmup Scheduler
# Goal: Build learning rate scheduler with warmup
# Expected outcome: Working warmup + decay scheduler

class WarmupLinearScheduler:
    """Linear warmup followed by linear decay"""
    
    def __init__(self, optimizer, warmup_steps, total_steps, min_lr=0):
        self.optimizer = optimizer
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.min_lr = min_lr
        self.base_lrs = [group['lr'] for group in optimizer.param_groups]
        self.current_step = 0
    
    def step(self):
        """Update learning rate"""
        self.current_step += 1
        
        if self.current_step < self.warmup_steps:
            # Warmup phase: linear increase
            lr_scale = self.current_step / self.warmup_steps
        else:
            # Decay phase: linear decrease
            progress = (self.current_step - self.warmup_steps) / (self.total_steps - self.warmup_steps)
            lr_scale = max(0, 1 - progress)
        
        for param_group, base_lr in zip(self.optimizer.param_groups, self.base_lrs):
            param_group['lr'] = self.min_lr + (base_lr - self.min_lr) * lr_scale
    
    def get_lr(self):
        """Get current learning rate"""
        return [group['lr'] for group in self.optimizer.param_groups]


class WarmupCosineScheduler:
    """Linear warmup followed by cosine annealing"""
    
    def __init__(self, optimizer, warmup_steps, total_steps, min_lr=0):
        self.optimizer = optimizer
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.min_lr = min_lr
        self.base_lrs = [group['lr'] for group in optimizer.param_groups]
        self.current_step = 0
    
    def step(self):
        """Update learning rate"""
        self.current_step += 1
        
        if self.current_step < self.warmup_steps:
            # Warmup phase
            lr_scale = self.current_step / self.warmup_steps
        else:
            # Cosine annealing phase
            progress = (self.current_step - self.warmup_steps) / (self.total_steps - self.warmup_steps)
            lr_scale = 0.5 * (1 + np.cos(np.pi * progress))
        
        for param_group, base_lr in zip(self.optimizer.param_groups, self.base_lrs):
            param_group['lr'] = self.min_lr + (base_lr - self.min_lr) * lr_scale
    
    def get_lr(self):
        """Get current learning rate"""
        return [group['lr'] for group in self.optimizer.param_groups]


# Visualize different schedulers
print("=" * 60)
print("可视化学习率调度")
print("=" * 60)

total_steps = 1000
warmup_steps = 100
base_lr = 0.001

# Create dummy optimizer
dummy_model = nn.Linear(1, 1)

schedulers = {
    'Warmup + Linear': WarmupLinearScheduler(
        optim.Adam(dummy_model.parameters(), lr=base_lr),
        warmup_steps, total_steps
    ),
    'Warmup + Cosine': WarmupCosineScheduler(
        optim.Adam(dummy_model.parameters(), lr=base_lr),
        warmup_steps, total_steps
    )
}

# Record learning rates
lr_history = {name: [] for name in schedulers.keys()}

for step in range(total_steps):
    for name, scheduler in schedulers.items():
        lr_history[name].append(scheduler.get_lr()[0])
        scheduler.step()

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Full schedule
for name, lrs in lr_history.items():
    axes[0].plot(lrs, label=name, linewidth=2)

axes[0].axvline(warmup_steps, color='red', linestyle='--', alpha=0.5, label='Warmup End')
axes[0].set_xlabel('Step', fontsize=12)
axes[0].set_ylabel('Learning Rate', fontsize=12)
axes[0].set_title('学习率调度对比', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: Warmup phase (zoomed in)
for name, lrs in lr_history.items():
    axes[1].plot(lrs[:warmup_steps*2], label=name, linewidth=2)

axes[1].axvline(warmup_steps, color='red', linestyle='--', alpha=0.5, label='Warmup End')
axes[1].set_xlabel('Step', fontsize=12)
axes[1].set_ylabel('Learning Rate', fontsize=12)
axes[1].set_title('Warmup 阶段（放大）', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n配置：")
print(f"  总步数: {total_steps}")
print(f"  Warmup 步数: {warmup_steps}")
print(f"  基础学习率: {base_lr}")
print(f"\n观察：")
print(f"  - Warmup 阶段学习率从 0 线性增加到最大值")
print(f"  - Linear decay 线性递减")
print(f"  - Cosine 平滑递减，后期衰减更慢")
print("\n✓ 学习率调度实现完成！")

## 5. 梯度累积 (Gradient Accumulation)

### 5.1 内存限制问题

**大模型训练的挑战**：
- GPU 内存有限（通常 16-80GB）
- 大模型占用大量内存
- 批次大小受限

**示例**：
- GPT-3 (175B 参数)：单个样本需要 ~700GB 内存
- BERT-Large：批次大小 8 需要 ~16GB 内存
- 小 GPU 可能只能用批次大小 1-2

### 5.2 梯度累积原理

**核心思想**：用多个小批次模拟一个大批次

**数学等价性**：

$$\nabla L_{\text{large batch}} = \frac{1}{B} \sum_{i=1}^B \nabla L(x_i)$$

$$= \frac{1}{k} \sum_{j=1}^k \left(\frac{1}{b} \sum_{i=1}^b \nabla L(x_{j,i})\right)$$

其中：
- $B = k \times b$：有效批次大小
- $k$：累积步数
- $b$：微批次大小

**算法流程**：

```python
for accumulation_step in range(k):
    # 前向传播（小批次）
    loss = model(micro_batch) / k
    
    # 反向传播（累积梯度）
    loss.backward()

# 更新参数（累积 k 步后）
optimizer.step()
optimizer.zero_grad()
```

### 5.3 优势与权衡

**优势**：
- ✅ 用有限内存训练大模型
- ✅ 模拟大批次的效果
- ✅ 更稳定的梯度估计
- ✅ 更好的泛化性能

**注意事项**：
- ⚠️ 训练时间增加（k 倍前向传播）
- ⚠️ 批归一化统计量基于微批次
- ⚠️ 需要调整学习率（通常与批次大小成正比）

**何时使用**：
- 模型太大，批次大小受限
- 需要大批次训练（如对比学习）
- GPU 内存不足

In [ ]:
# 🔬 Micro Practice: Implement Gradient Accumulation
# Goal: Demonstrate gradient accumulation for memory efficiency
# Expected outcome: Show equivalence to large batch training

print("=" * 60)
print("梯度累积演示")
print("=" * 60)

# Create model and data
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(10, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    
    def forward(self, x):
        return self.fc(x)

# Generate data
torch.manual_seed(42)
X = torch.randn(128, 10)
y = torch.randn(128, 1)

# Training function without gradient accumulation
def train_normal(model, X, y, batch_size, num_epochs):
    """Normal training with large batch"""
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.MSELoss()
    
    losses = []
    for epoch in range(num_epochs):
        epoch_loss = 0
        for i in range(0, len(X), batch_size):
            batch_X = X[i:i+batch_size]
            batch_y = y[i:i+batch_size]
            
            optimizer.zero_grad()
            output = model(batch_X)
            loss = criterion(output, batch_y)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
        
        losses.append(epoch_loss / (len(X) // batch_size))
    
    return losses

# Training function with gradient accumulation
def train_with_accumulation(model, X, y, micro_batch_size, accumulation_steps, num_epochs):
    """Training with gradient accumulation"""
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.MSELoss()
    
    effective_batch_size = micro_batch_size * accumulation_steps
    
    losses = []
    for epoch in range(num_epochs):
        epoch_loss = 0
        num_batches = 0
        
        for i in range(0, len(X), effective_batch_size):
            optimizer.zero_grad()
            
            # Accumulate gradients over multiple micro-batches
            accumulated_loss = 0
            for j in range(accumulation_steps):
                start_idx = i + j * micro_batch_size
                end_idx = start_idx + micro_batch_size
                
                if end_idx > len(X):
                    break
                
                batch_X = X[start_idx:end_idx]
                batch_y = y[start_idx:end_idx]
                
                output = model(batch_X)
                loss = criterion(output, batch_y)
                
                # Scale loss by accumulation steps
                (loss / accumulation_steps).backward()
                accumulated_loss += loss.item()
            
            # Update after accumulation
            optimizer.step()
            
            epoch_loss += accumulated_loss / accumulation_steps
            num_batches += 1
        
        losses.append(epoch_loss / num_batches)
    
    return losses

# Compare different configurations
configs = [
    {'name': 'Batch 32 (Normal)', 'batch_size': 32, 'use_accumulation': False},
    {'name': 'Batch 8 × 4 steps (Accumulation)', 'micro_batch': 8, 'accum_steps': 4, 'use_accumulation': True},
]

results = {}
num_epochs = 50

for config in configs:
    model = SimpleNet()
    
    if config['use_accumulation']:
        losses = train_with_accumulation(
            model, X, y, 
            config['micro_batch'], 
            config['accum_steps'], 
            num_epochs
        )
    else:
        losses = train_normal(model, X, y, config['batch_size'], num_epochs)
    
    results[config['name']] = losses
    print(f"{config['name']:40s} - 最终损失: {losses[-1]:.6f}")

# Visualize
plt.figure(figsize=(12, 6))
for name, losses in results.items():
    plt.plot(losses, label=name, linewidth=2, marker='o', markersize=4, alpha=0.7)

plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('梯度累积 vs 正常训练', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n关键观察：")
print("- 有效批次大小相同时，两种方法收敛曲线几乎相同")
print("- 梯度累积允许用小内存模拟大批次训练")
print("- 训练时间会增加（更多前向/反向传播）")
print("\n内存节省估算：")
print(f"  正常训练 (batch=32): ~32x 内存")
print(f"  梯度累积 (batch=8): ~8x 内存 (节省 75%)")
print("\n✓ 梯度累积实现完成！")

## 6. 混合精度训练 (Mixed Precision Training)

### 6.1 浮点精度回顾

**浮点数表示**：

| 类型 | 位数 | 符号 | 指数 | 尾数 | 范围 |
|------|------|------|------|------|------|
| **FP32** | 32 | 1 | 8 | 23 | ±3.4×10³⁸ |
| **FP16** | 16 | 1 | 5 | 10 | ±6.5×10⁴ |
| **BF16** | 16 | 1 | 8 | 7 | ±3.4×10³⁸ |

**FP16 的问题**：
- 范围小：容易上溢/下溢
- 精度低：小梯度可能变为 0

**BF16 的优势**：
- 范围与 FP32 相同
- 更适合深度学习
- 但精度比 FP16 低

### 6.2 混合精度训练原理

**核心思想**：在不同阶段使用不同精度

**训练流程**：

1. **权重存储**：FP32（主副本）
2. **前向传播**：FP16（快速计算）
3. **损失计算**：FP16
4. **损失缩放**：放大损失防止下溢
5. **反向传播**：FP16（快速计算）
6. **梯度缩放**：缩小梯度到正常范围
7. **权重更新**：FP32（精确更新）

**损失缩放（Loss Scaling）**：

$$L_{\text{scaled}} = L \times S$$
$$\nabla_{\text{scaled}} = \nabla L \times S$$
$$\nabla = \nabla_{\text{scaled}} / S$$

其中 $S$ 是缩放因子（通常 512-65536）

### 6.3 优势与挑战

**优势**：
- ✅ **速度提升**：2-3倍加速（Tensor Core）
- ✅ **内存节省**：减少 50% 内存占用
- ✅ **吞吐量提升**：可以用更大批次
- ✅ **几乎无精度损失**：正确实现时

**挑战**：
- ⚠️ 需要损失缩放
- ⚠️ 某些操作必须用 FP32（如 softmax）
- ⚠️ 需要硬件支持（Volta 及以上）

**何时使用**：
- 训练大模型
- GPU 支持 Tensor Core
- 需要加速训练
- 内存受限

### 6.4 自动混合精度（AMP）

PyTorch 提供 `torch.cuda.amp` 自动处理：
- 自动选择精度
- 自动损失缩放
- 动态调整缩放因子

In [ ]:
# 🔬 Micro Practice: Implement Mixed Precision Training
# Goal: Demonstrate automatic mixed precision (AMP)
# Expected outcome: Compare FP32 vs FP16 training

print("=" * 60)
print("混合精度训练演示")
print("=" * 60)

# Create a larger model for better demonstration
class LargerNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(100, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )
    
    def forward(self, x):
        return self.layers(x)

# Generate larger dataset
torch.manual_seed(42)
X_train = torch.randn(1000, 100)
y_train = torch.randint(0, 10, (1000,))

# Training with FP32
def train_fp32(model, X, y, num_epochs, batch_size=32):
    """Standard FP32 training"""
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    
    start_time = time.time()
    losses = []
    
    for epoch in range(num_epochs):
        epoch_loss = 0
        for i in range(0, len(X), batch_size):
            batch_X = X[i:i+batch_size].to(device)
            batch_y = y[i:i+batch_size].to(device)
            
            optimizer.zero_grad()
            output = model(batch_X)
            loss = criterion(output, batch_y)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
        
        losses.append(epoch_loss / (len(X) // batch_size))
    
    training_time = time.time() - start_time
    return losses, training_time

# Training with AMP (FP16)
def train_amp(model, X, y, num_epochs, batch_size=32):
    """Mixed precision training with AMP"""
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    scaler = GradScaler()
    
    start_time = time.time()
    losses = []
    
    for epoch in range(num_epochs):
        epoch_loss = 0
        for i in range(0, len(X), batch_size):
            batch_X = X[i:i+batch_size].to(device)
            batch_y = y[i:i+batch_size].to(device)
            
            optimizer.zero_grad()
            
            # Forward pass with autocast
            with autocast():
                output = model(batch_X)
                loss = criterion(output, batch_y)
            
            # Backward pass with gradient scaling
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            epoch_loss += loss.item()
        
        losses.append(epoch_loss / (len(X) // batch_size))
    
    training_time = time.time() - start_time
    return losses, training_time

# Compare FP32 vs AMP
print("\n训练配置：")
print(f"  数据集大小: {len(X_train)}")
print(f"  批次大小: 32")
print(f"  训练轮数: 20")
print(f"  设备: {device}")

num_epochs = 20

# FP32 training
print("\n1. FP32 训练...")
model_fp32 = LargerNet()
losses_fp32, time_fp32 = train_fp32(model_fp32, X_train, y_train, num_epochs)
print(f"   训练时间: {time_fp32:.2f}s")
print(f"   最终损失: {losses_fp32[-1]:.4f}")

# AMP training
if device.type == 'cuda':
    print("\n2. 混合精度 (AMP) 训练...")
    model_amp = LargerNet()
    losses_amp, time_amp = train_amp(model_amp, X_train, y_train, num_epochs)
    print(f"   训练时间: {time_amp:.2f}s")
    print(f"   最终损失: {losses_amp[-1]:.4f}")
    print(f"   加速比: {time_fp32/time_amp:.2f}x")
    
    # Visualize
    plt.figure(figsize=(12, 5))
    
    # Plot 1: Loss curves
    plt.subplot(1, 2, 1)
    plt.plot(losses_fp32, label='FP32', linewidth=2)
    plt.plot(losses_amp, label='AMP (FP16)', linewidth=2, linestyle='--')
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.title('训练损失对比', fontsize=14, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Plot 2: Training time
    plt.subplot(1, 2, 2)
    methods = ['FP32', 'AMP']
    times = [time_fp32, time_amp]
    colors = ['steelblue', 'orange']
    bars = plt.bar(methods, times, color=colors, alpha=0.7)
    plt.ylabel('训练时间 (秒)', fontsize=12)
    plt.title('训练速度对比', fontsize=14, fontweight='bold')
    plt.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for bar, time_val in zip(bars, times):
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height,
                f'{time_val:.2f}s',
                ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print("\n性能总结：")
    print(f"  内存节省: ~50%")
    print(f"  速度提升: {time_fp32/time_amp:.2f}x")
    print(f"  精度损失: {abs(losses_fp32[-1] - losses_amp[-1]):.6f} (可忽略)")
else:
    print("\n注意：混合精度训练需要 CUDA GPU")
    print("当前在 CPU 上运行，跳过 AMP 演示")

print("\n✓ 混合精度训练演示完成！")

## 7. 梯度裁剪和训练稳定性 (Gradient Clipping & Stability)

### 7.1 梯度爆炸问题

**什么是梯度爆炸？**
- 梯度在反向传播中指数级增长
- 导致参数更新过大
- 训练发散，损失变为 NaN

**常见原因**：
- 深层网络（RNN, Transformer）
- 不当的初始化
- 学习率过大
- 数值不稳定

**症状**：
- 损失突然激增
- 梯度范数非常大（>100）
- 参数变为 NaN 或 Inf

### 7.2 梯度裁剪策略

**1. 按值裁剪（Clip by Value）**

$$g_i = \text{clip}(g_i, -\theta, \theta)$$

- 简单直接
- 可能改变梯度方向

**2. 按范数裁剪（Clip by Norm）** ⭐ 推荐

$$g = \begin{cases}
g & \text{if } \|g\| \leq \theta \\
\theta \cdot \frac{g}{\|g\|} & \text{if } \|g\| > \theta
\end{cases}$$

- 保持梯度方向
- 只缩放大小
- Transformer 训练的标准做法

**3. 按全局范数裁剪（Clip by Global Norm）**

$$g_{\text{all}} = \sqrt{\sum_i \|g_i\|^2}$$
$$g_i = g_i \cdot \min\left(1, \frac{\theta}{g_{\text{all}}}\right)$$

- 考虑所有参数的梯度
- 更稳定

**裁剪阈值选择**：
- RNN: 1.0 - 5.0
- Transformer: 1.0
- CNN: 通常不需要

### 7.3 其他稳定性技巧

**1. 层归一化（Layer Normalization）**
- 稳定激活值分布
- 减少内部协变量偏移
- Transformer 的关键组件

**2. 权重初始化**
- Xavier/Glorot 初始化
- He 初始化
- 防止梯度消失/爆炸

**3. 残差连接（Residual Connections）**
- 提供梯度直通路径
- 缓解梯度消失
- 允许训练更深网络

**4. 梯度检查点（Gradient Checkpointing）**
- 用计算换内存
- 重新计算中间激活
- 训练超大模型

In [ ]:
# 🔬 Micro Practice: Implement Gradient Clipping
# Goal: Demonstrate gradient clipping for training stability
# Expected outcome: Show how clipping prevents gradient explosion

print("=" * 60)
print("梯度裁剪演示")
print("=" * 60)

# Create a deep network prone to gradient explosion
class DeepNet(nn.Module):
    def __init__(self, num_layers=10):
        super().__init__()
        layers = []
        for i in range(num_layers):
            layers.append(nn.Linear(50, 50))
            layers.append(nn.Tanh())  # Tanh can cause gradient issues
        layers.append(nn.Linear(50, 1))
        self.layers = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.layers(x)

# Generate data
torch.manual_seed(42)
X = torch.randn(200, 50)
y = torch.randn(200, 1)

# Training without gradient clipping
def train_without_clipping(model, X, y, lr=0.01, num_epochs=50):
    """Training without gradient clipping"""
    optimizer = optim.SGD(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    
    losses = []
    grad_norms = []
    
    for epoch in range(num_epochs):
        optimizer.zero_grad()
        output = model(X)
        loss = criterion(output, y)
        
        if torch.isnan(loss) or torch.isinf(loss):
            print(f"   训练在 epoch {epoch} 发散！")
            break
        
        loss.backward()
        
        # Record gradient norm
        total_norm = 0
        for p in model.parameters():
            if p.grad is not None:
                param_norm = p.grad.data.norm(2)
                total_norm += param_norm.item() ** 2
        total_norm = total_norm ** 0.5
        grad_norms.append(total_norm)
        
        optimizer.step()
        losses.append(loss.item())
    
    return losses, grad_norms

# Training with gradient clipping
def train_with_clipping(model, X, y, lr=0.01, num_epochs=50, max_norm=1.0):
    """Training with gradient clipping"""
    optimizer = optim.SGD(model.parameters(), lr=lr)
    criterion = nn.MSELoss()
    
    losses = []
    grad_norms = []
    
    for epoch in range(num_epochs):
        optimizer.zero_grad()
        output = model(X)
        loss = criterion(output, y)
        loss.backward()
        
        # Clip gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)
        
        # Record gradient norm (after clipping)
        total_norm = 0
        for p in model.parameters():
            if p.grad is not None:
                param_norm = p.grad.data.norm(2)
                total_norm += param_norm.item() ** 2
        total_norm = total_norm ** 0.5
        grad_norms.append(total_norm)
        
        optimizer.step()
        losses.append(loss.item())
    
    return losses, grad_norms

# Compare with and without clipping
print("\n1. 不使用梯度裁剪...")
model1 = DeepNet(num_layers=10)
losses1, grad_norms1 = train_without_clipping(model1, X, y, lr=0.1)
print(f"   完成 {len(losses1)} 个 epoch")
if len(losses1) > 0:
    print(f"   最终损失: {losses1[-1]:.4f}")
    print(f"   最大梯度范数: {max(grad_norms1):.2f}")

print("\n2. 使用梯度裁剪 (max_norm=1.0)...")
model2 = DeepNet(num_layers=10)
losses2, grad_norms2 = train_with_clipping(model2, X, y, lr=0.1, max_norm=1.0)
print(f"   完成 {len(losses2)} 个 epoch")
print(f"   最终损失: {losses2[-1]:.4f}")
print(f"   最大梯度范数: {max(grad_norms2):.2f}")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Loss curves
axes[0].plot(losses1, label='无裁剪', linewidth=2, color='red', alpha=0.7)
axes[0].plot(losses2, label='有裁剪 (max_norm=1.0)', linewidth=2, color='green', alpha=0.7)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('训练损失对比', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_yscale('log')

# Plot 2: Gradient norms
axes[1].plot(grad_norms1, label='无裁剪', linewidth=2, color='red', alpha=0.7)
axes[1].plot(grad_norms2, label='有裁剪 (max_norm=1.0)', linewidth=2, color='green', alpha=0.7)
axes[1].axhline(y=1.0, color='black', linestyle='--', alpha=0.5, label='裁剪阈值')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('梯度范数', fontsize=12)
axes[1].set_title('梯度范数对比', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

print("\n关键观察：")
print("- 无裁剪：梯度爆炸导致训练不稳定或发散")
print("- 有裁剪：梯度被限制在合理范围，训练稳定")
print("- 裁剪保持梯度方向，只缩放大小")
print("\n最佳实践：")
print("- RNN/LSTM: max_norm = 1.0 - 5.0")
print("- Transformer: max_norm = 1.0")
print("- 监控梯度范数，调整阈值")
print("\n✓ 梯度裁剪演示完成！")

## 8. 综合训练流程 (Complete Training Pipeline)

### 8.1 整合所有技术

**现代大模型训练的标准配置**：

```python
# 1. 优化器：AdamW
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    betas=(0.9, 0.999),
    weight_decay=0.01
)

# 2. 学习率调度：Warmup + Cosine
scheduler = WarmupCosineScheduler(
    optimizer,
    warmup_steps=10000,
    total_steps=100000
)

# 3. 梯度累积
accumulation_steps = 4

# 4. 混合精度
scaler = GradScaler()

# 5. 梯度裁剪
max_grad_norm = 1.0
```

### 8.2 完整训练循环

**训练步骤**：

1. **数据加载**：小批次数据
2. **前向传播**：使用 autocast (FP16)
3. **损失计算**：缩放损失
4. **反向传播**：累积梯度
5. **梯度裁剪**：防止爆炸
6. **参数更新**：每 k 步更新一次
7. **学习率调度**：动态调整学习率
8. **监控指标**：损失、梯度范数、学习率

### 8.3 监控和日志

**关键指标**：

1. **训练损失**：主要优化目标
2. **验证损失**：泛化性能
3. **学习率**：当前学习率
4. **梯度范数**：检测梯度爆炸
5. **内存使用**：GPU 内存占用
6. **训练速度**：samples/sec 或 tokens/sec

**可视化**：
- 实时损失曲线
- 学习率变化
- 梯度范数分布
- 内存使用趋势

In [ ]:
# 🔬 Micro Practice: Complete Training Pipeline
# Goal: Integrate all optimization techniques
# Expected outcome: Production-ready training loop

print("=" * 60)
print("完整训练流程演示")
print("=" * 60)

class ComprehensiveTrainer:
    """Complete training pipeline with all optimizations"""
    
    def __init__(self, model, train_data, val_data, config):
        self.model = model.to(device)
        self.train_data = train_data
        self.val_data = val_data
        self.config = config
        
        # Optimizer: AdamW
        self.optimizer = optim.AdamW(
            model.parameters(),
            lr=config['lr'],
            betas=(0.9, 0.999),
            weight_decay=config['weight_decay']
        )
        
        # Scheduler: Warmup + Cosine
        self.scheduler = WarmupCosineScheduler(
            self.optimizer,
            warmup_steps=config['warmup_steps'],
            total_steps=config['total_steps']
        )
        
        # Mixed precision
        self.use_amp = config.get('use_amp', False) and device.type == 'cuda'
        if self.use_amp:
            self.scaler = GradScaler()
        
        # Gradient accumulation
        self.accumulation_steps = config['accumulation_steps']
        
        # Gradient clipping
        self.max_grad_norm = config['max_grad_norm']
        
        # Metrics
        self.history = {
            'train_loss': [],
            'val_loss': [],
            'learning_rate': [],
            'grad_norm': []
        }
    
    def train_epoch(self, epoch):
        """Train for one epoch"""
        self.model.train()
        epoch_loss = 0
        num_batches = 0
        
        X_train, y_train = self.train_data
        batch_size = self.config['batch_size']
        
        for i in range(0, len(X_train), batch_size * self.accumulation_steps):
            self.optimizer.zero_grad()
            accumulated_loss = 0
            
            # Gradient accumulation loop
            for j in range(self.accumulation_steps):
                start_idx = i + j * batch_size
                end_idx = start_idx + batch_size
                
                if end_idx > len(X_train):
                    break
                
                batch_X = X_train[start_idx:end_idx].to(device)
                batch_y = y_train[start_idx:end_idx].to(device)
                
                # Forward pass with mixed precision
                if self.use_amp:
                    with autocast():
                        output = self.model(batch_X)
                        loss = nn.functional.cross_entropy(output, batch_y)
                        loss = loss / self.accumulation_steps
                    
                    self.scaler.scale(loss).backward()
                else:
                    output = self.model(batch_X)
                    loss = nn.functional.cross_entropy(output, batch_y)
                    loss = loss / self.accumulation_steps
                    loss.backward()
                
                accumulated_loss += loss.item()
            
            # Gradient clipping
            if self.use_amp:
                self.scaler.unscale_(self.optimizer)
            
            grad_norm = torch.nn.utils.clip_grad_norm_(
                self.model.parameters(), 
                self.max_grad_norm
            )
            
            # Optimizer step
            if self.use_amp:
                self.scaler.step(self.optimizer)
                self.scaler.update()
            else:
                self.optimizer.step()
            
            # Scheduler step
            self.scheduler.step()
            
            # Record metrics
            epoch_loss += accumulated_loss
            num_batches += 1
            self.history['grad_norm'].append(grad_norm.item())
            self.history['learning_rate'].append(self.scheduler.get_lr()[0])
        
        avg_loss = epoch_loss / num_batches if num_batches > 0 else 0
        self.history['train_loss'].append(avg_loss)
        return avg_loss
    
    def validate(self):
        """Validate the model"""
        self.model.eval()
        val_loss = 0
        num_batches = 0
        
        X_val, y_val = self.val_data
        batch_size = self.config['batch_size']
        
        with torch.no_grad():
            for i in range(0, len(X_val), batch_size):
                batch_X = X_val[i:i+batch_size].to(device)
                batch_y = y_val[i:i+batch_size].to(device)
                
                output = self.model(batch_X)
                loss = nn.functional.cross_entropy(output, batch_y)
                
                val_loss += loss.item()
                num_batches += 1
        
        avg_loss = val_loss / num_batches if num_batches > 0 else 0
        self.history['val_loss'].append(avg_loss)
        return avg_loss
    
    def train(self, num_epochs):
        """Complete training loop"""
        print(f"\n开始训练...")
        print(f"配置: {self.config}")
        print("-" * 60)
        
        for epoch in range(num_epochs):
            train_loss = self.train_epoch(epoch)
            val_loss = self.validate()
            
            print(f"Epoch {epoch+1}/{num_epochs} - "
                  f"Train Loss: {train_loss:.4f}, "
                  f"Val Loss: {val_loss:.4f}, "
                  f"LR: {self.scheduler.get_lr()[0]:.6f}")
        
        print("-" * 60)
        print("训练完成！")
        return self.history

# Prepare data
torch.manual_seed(42)
X_train = torch.randn(800, 50)
y_train = torch.randint(0, 10, (800,))
X_val = torch.randn(200, 50)
y_val = torch.randint(0, 10, (200,))

# Model
model = nn.Sequential(
    nn.Linear(50, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 10)
)

# Training configuration
config = {
    'lr': 1e-3,
    'weight_decay': 0.01,
    'warmup_steps': 20,
    'total_steps': 200,
    'batch_size': 32,
    'accumulation_steps': 2,
    'max_grad_norm': 1.0,
    'use_amp': device.type == 'cuda'
}

# Train
trainer = ComprehensiveTrainer(
    model, 
    (X_train, y_train), 
    (X_val, y_val), 
    config
)
history = trainer.train(num_epochs=20)

print("\n✓ 完整训练流程演示完成！")

In [ ]:
# Visualize training metrics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Loss curves
axes[0, 0].plot(history['train_loss'], label='Train Loss', linewidth=2)
axes[0, 0].plot(history['val_loss'], label='Val Loss', linewidth=2)
axes[0, 0].set_xlabel('Epoch', fontsize=12)
axes[0, 0].set_ylabel('Loss', fontsize=12)
axes[0, 0].set_title('训练和验证损失', fontsize=14, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Learning rate schedule
axes[0, 1].plot(history['learning_rate'], linewidth=2, color='orange')
axes[0, 1].set_xlabel('Step', fontsize=12)
axes[0, 1].set_ylabel('Learning Rate', fontsize=12)
axes[0, 1].set_title('学习率调度', fontsize=14, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Gradient norms
axes[1, 0].plot(history['grad_norm'], linewidth=1, alpha=0.7, color='green')
axes[1, 0].axhline(y=config['max_grad_norm'], color='red', linestyle='--', 
                   alpha=0.5, label='Clipping Threshold')
axes[1, 0].set_xlabel('Step', fontsize=12)
axes[1, 0].set_ylabel('Gradient Norm', fontsize=12)
axes[1, 0].set_title('梯度范数', fontsize=14, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Training summary
axes[1, 1].axis('off')
summary_text = f"""
训练总结
{'='*40}

配置：
  优化器: AdamW
  学习率: {config['lr']}
  权重衰减: {config['weight_decay']}
  批次大小: {config['batch_size']}
  累积步数: {config['accumulation_steps']}
  有效批次: {config['batch_size'] * config['accumulation_steps']}
  梯度裁剪: {config['max_grad_norm']}
  混合精度: {'是' if config['use_amp'] else '否'}

结果：
  最终训练损失: {history['train_loss'][-1]:.4f}
  最终验证损失: {history['val_loss'][-1]:.4f}
  最佳验证损失: {min(history['val_loss']):.4f}
  
技术栈：
  ✓ AdamW 优化器
  ✓ Warmup + Cosine 调度
  ✓ 梯度累积
  ✓ 梯度裁剪
  {'✓ 混合精度训练' if config['use_amp'] else '✗ 混合精度 (CPU)'}
"""
axes[1, 1].text(0.1, 0.5, summary_text, fontsize=10, family='monospace',
                verticalalignment='center')

plt.tight_layout()
plt.show()

print("\n训练指标分析：")
print(f"  损失下降: {(history['train_loss'][0] - history['train_loss'][-1]) / history['train_loss'][0] * 100:.1f}%")
print(f"  学习率范围: {min(history['learning_rate']):.6f} - {max(history['learning_rate']):.6f}")
print(f"  平均梯度范数: {np.mean(history['grad_norm']):.4f}")
print(f"  最大梯度范数: {max(history['grad_norm']):.4f}")

## 9. 常见问题 (FAQ)

### Q1: 训练 Transformer 应该用哪个优化器？

**A:** AdamW 是首选
- **原因**：自适应学习率 + 解耦权重衰减
- **配置**：lr=1e-4 到 5e-4, weight_decay=0.01
- **替代**：Adam（但泛化性能稍差）
- **不推荐**：SGD（需要精心调参，收敛慢）

### Q2: 如何选择 Warmup 步数？

**A:** 取决于模型大小和总训练步数
- **小模型**：1000-2000 步
- **中等模型（BERT-Base）**：10000 步
- **大模型（GPT-3）**：40000 步
- **经验法则**：总步数的 1-10%
- **观察**：如果早期损失不稳定，增加 warmup

### Q3: 什么时候使用梯度累积？

**A:** 当批次大小受内存限制时
- **场景 1**：模型太大，只能用小批次
- **场景 2**：需要大批次训练（如对比学习）
- **场景 3**：多 GPU 训练时平衡负载
- **注意**：训练时间会增加
- **建议**：有效批次大小 = 微批次 × 累积步数

### Q4: FP16 和 BF16 哪个更好？

**A:** 取决于硬件和任务
- **FP16**：
  - 更快（Volta 及以上 GPU）
  - 需要损失缩放
  - 范围小，容易溢出
- **BF16**：
  - 范围与 FP32 相同
  - 不需要损失缩放
  - 需要 Ampere 及以上 GPU
  - **推荐用于 Transformer**

### Q5: 梯度裁剪的阈值如何选择？

**A:** 根据模型类型和观察梯度范数
- **RNN/LSTM**：1.0 - 5.0
- **Transformer**：1.0
- **CNN**：通常不需要
- **方法**：先不裁剪，观察梯度范数分布，设置为 95 分位数

### Q6: 学习率应该设置多大？

**A:** 取决于优化器和批次大小
- **Adam/AdamW**：1e-4 到 5e-4
- **SGD**：0.01 到 0.1
- **规则**：学习率 ∝ √批次大小
- **调参**：使用学习率查找（LR Finder）

### Q7: 如何判断训练是否稳定？

**A:** 监控以下指标
- ✅ 损失平滑下降
- ✅ 梯度范数在合理范围（0.1-10）
- ✅ 学习率按预期变化
- ❌ 损失突然激增 → 学习率太大或梯度爆炸
- ❌ 损失不下降 → 学习率太小或模型问题
- ❌ 梯度范数 > 100 → 需要梯度裁剪

### Q8: 混合精度训练会损失精度吗？

**A:** 正确实现时几乎不会
- **关键**：使用损失缩放
- **某些操作保持 FP32**：softmax, layer norm
- **主权重保持 FP32**：只有前向/反向用 FP16
- **实践**：精度差异 < 0.1%

### Q9: 如何加速训练？

**A:** 多种方法组合
1. **混合精度**：2-3x 加速
2. **梯度累积**：允许更大批次
3. **梯度检查点**：节省内存，用更大模型
4. **数据并行**：多 GPU 训练
5. **模型并行**：超大模型
6. **优化数据加载**：预处理、缓存

### Q10: 训练大模型的最佳实践？

**A:** 综合使用所有技术
```python
# 标准配置
optimizer = AdamW(lr=1e-4, weight_decay=0.01)
scheduler = WarmupCosineScheduler(warmup=10000)
accumulation_steps = 4
max_grad_norm = 1.0
use_amp = True  # FP16/BF16
```

**监控**：
- 每 100 步记录损失
- 每 1000 步验证
- 每 5000 步保存检查点
- 实时可视化指标

## 10. 总结 (Summary)

### 10.1 本章回顾

**你已经学会了**：

✅ **自适应优化器**
- Adam 和 AdamW 的原理
- 从零实现 Adam
- 不同优化器的对比

✅ **学习率调度**
- Warmup 的重要性
- 线性和余弦衰减
- 实现自定义调度器

✅ **梯度累积**
- 用小内存模拟大批次
- 数学等价性
- 内存节省 75%

✅ **混合精度训练**
- FP16/BF16 原理
- 自动混合精度（AMP）
- 2-3x 训练加速

✅ **梯度裁剪**
- 防止梯度爆炸
- 按范数裁剪
- 训练稳定性技巧

✅ **完整训练流程**
- 整合所有技术
- 生产级训练循环
- 监控和可视化

### 10.2 关键要点

**优化器选择**：
- Transformer → AdamW
- CNN → SGD + Momentum
- RNN → Adam

**学习率策略**：
- 使用 Warmup（大模型必需）
- Cosine 衰减效果好
- 监控学习率变化

**内存优化**：
- 梯度累积：节省内存
- 混合精度：节省 50% 内存
- 梯度检查点：用计算换内存

**训练稳定性**：
- 梯度裁剪：防止爆炸
- 层归一化：稳定激活
- 残差连接：缓解梯度消失

### 10.3 最佳实践清单

**训练前**：
- ✅ 选择合适的优化器（AdamW）
- ✅ 设置学习率调度（Warmup + Cosine）
- ✅ 配置梯度累积（如果内存不足）
- ✅ 启用混合精度（如果有 GPU）
- ✅ 设置梯度裁剪（max_norm=1.0）

**训练中**：
- ✅ 监控损失曲线
- ✅ 检查梯度范数
- ✅ 观察学习率变化
- ✅ 定期验证
- ✅ 保存检查点

**训练后**：
- ✅ 分析训练曲线
- ✅ 对比不同配置
- ✅ 记录最佳超参数
- ✅ 评估最终模型

### 10.4 Transformer 训练标准配置

```python
# 推荐配置
config = {
    # 优化器
    'optimizer': 'AdamW',
    'lr': 1e-4,
    'betas': (0.9, 0.999),
    'weight_decay': 0.01,
    
    # 学习率调度
    'warmup_steps': 10000,
    'scheduler': 'cosine',
    
    # 批次和累积
    'batch_size': 32,
    'accumulation_steps': 4,  # 有效批次 = 128
    
    # 稳定性
    'max_grad_norm': 1.0,
    
    # 性能
    'use_amp': True,  # FP16/BF16
    'gradient_checkpointing': False,  # 超大模型时启用
}
```

### 10.5 下一步

**Module 7: 分布式训练**
- 数据并行（DataParallel, DistributedDataParallel）
- 模型并行（Pipeline, Tensor Parallelism）
- ZeRO 优化器
- 大规模训练技巧

**推荐阅读**：
- [Adam: A Method for Stochastic Optimization](https://arxiv.org/abs/1412.6980)
- [Decoupled Weight Decay Regularization](https://arxiv.org/abs/1711.05101) (AdamW)
- [Mixed Precision Training](https://arxiv.org/abs/1710.03740)
- [Scaling Laws for Neural Language Models](https://arxiv.org/abs/2001.08361)

---

**🎉 恭喜完成高级优化技术！**

你现在掌握了训练大模型的核心技术，可以高效、稳定地训练 Transformer 模型了！💪

## 思考题参考答案

### 问题 1：训练 Transformer 应该用哪个优化器，为什么 Adam 比 SGD 更适合 Transformer？

**答：AdamW 是 Transformer 训练的首选优化器。**

Adam 更适合 Transformer 的原因如下：

**1. 自适应学习率**

Adam 为每个参数独立维护学习率：

$$\theta_{t+1} = \theta_t - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

Transformer 不同层（Embedding、Attention、FFN）的梯度尺度差异极大。Adam 通过二阶矩 $\hat{v}_t$ 自动缩放每个参数的学习率，使不同层都能以合适的步长更新，而 SGD 使用统一学习率，难以同时兼顾所有层。

**2. 对稀疏梯度友好**

Transformer 的 Embedding 层存在大量稀疏梯度（只有当前 token 的梯度非零）。Adam 的自适应机制使得稀疏更新也能有效传播，SGD+Momentum 在这方面表现较差。

**3. AdamW 解耦权重衰减**

标准 Adam 中 L2 正则化与自适应学习率相互干扰，AdamW 将权重衰减从梯度中解耦：

$$\theta_{t+1} = \theta_t - \eta \left(\frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon} + \lambda \theta_t\right)$$

这使得正则化效果更稳定，泛化性能更好，是 Transformer 的标准配置（lr=1e-4~5e-4，weight_decay=0.01）。

**4. 实践推荐**

| 场景 | 优化器 | 理由 |
|------|--------|------|
| Transformer | AdamW | 最佳泛化，自适应学习率 |
| CNN | SGD+Momentum | 更好的最终精度 |
| RNN | Adam | 缓解梯度消失 |

---

### 问题 2：如何选择学习率调度策略（Warmup 步数、Cosine vs Linear 衰减）？

**答：根据模型规模和任务性质综合选择。**

**Warmup 的必要性**

大模型训练初期，随机初始化的参数导致梯度方差大，Adam 的一/二阶矩估计也不准确（偏差修正项 $1-\beta^t$ 在 $t$ 小时影响显著），过大的学习率容易导致损失发散。Warmup 让学习率从 0 线性增大到最大值，稳定训练初期。

**Warmup 步数选择**

$$T_{\text{warmup}} \approx (1\% \sim 10\%) \times T_{\text{total}}$$

| 模型规模 | Warmup 步数 |
|---------|------------|
| 小模型 (<100M) | 1,000~2,000 |
| 中等模型 (BERT-Base 110M) | 10,000 |
| 大模型 (GPT-3 175B) | 40,000 |

**Cosine vs Linear 衰减**

- **Linear 衰减**：$\eta_t = \eta_{\max}(1 - \frac{t - T_w}{T - T_w})$，简单直接，均匀递减至 0
- **Cosine 衰减**：$\eta_t = \eta_{\min} + \frac{1}{2}(\eta_{\max} - \eta_{\min})(1 + \cos\frac{(t-T_w)\pi}{T - T_w})$，后期衰减更缓慢

**推荐**：Cosine 衰减通常效果更好，因为后期小学习率有充足时间精细调整参数，收敛到更好的极小值。大多数预训练语言模型（BERT、GPT 系列）均采用 Warmup + Cosine 策略。

---

### 问题 3：梯度累积如何节省内存，其数学等价性是什么？

**答：梯度累积通过将一个大批次拆分为多个小批次来节省内存，同时保持等效的优化效果。**

**内存节省原理**

训练时显存的主要消耗之一是批次中每个样本的激活值（前向传播的中间结果）。假设目标有效批次大小为 $B$，若 GPU 内存只能容纳 $b$ 个样本（$b \ll B$），则：

- 标准训练：一次前向传播批次 $B$，激活值占用 $O(B)$ 内存
- 梯度累积：每次只处理批次 $b$，累积 $k = B/b$ 步后再更新，激活值只占用 $O(b)$ 内存

**数学等价性**

对于损失函数 $L$ 满足批次可加性时，梯度累积与直接大批次训练在期望上等价：

$$\nabla L_{\text{大批次}} = \frac{1}{B}\sum_{i=1}^{B} \nabla \ell(x_i) = \frac{1}{k}\sum_{j=1}^{k}\left(\frac{1}{b}\sum_{i=1}^{b}\nabla \ell(x_{j,i})\right)$$

其中每个小批次损失需除以累积步数 $k$（等比缩放）以保证梯度尺度一致。

**注意事项**

- BatchNorm 统计量仍基于小批次计算，与真实大批次有差异（建议改用 LayerNorm）
- 学习率通常应随有效批次大小线性缩放：$\eta_{\text{eff}} = \eta \times k$
- 训练时间约增加 $k$ 倍（但显存节省约 $k$ 倍）

---

### 问题 4：混合精度训练为什么更快，损失缩放（Loss Scaling）如何解决 FP16 精度问题？

**答：混合精度训练结合 FP16 的计算速度优势与 FP32 的数值稳定性，通过损失缩放防止梯度下溢。**

**为什么更快**

| 精度 | 位宽 | 带宽占用 | Tensor Core 加速 |
|------|------|----------|-----------------|
| FP32 | 32 bit | 基准 | 不支持（旧架构）|
| FP16 | 16 bit | 50% | 2-8x 加速 |
| BF16 | 16 bit | 50% | 2-8x 加速 |

速度提升来自两方面：
1. **减少内存带宽**：FP16 参数/激活值体积减半，内存读写更快
2. **Tensor Core 加速**：Volta 及以上 GPU 的 Tensor Core 专门优化 FP16 矩阵乘法，吞吐量显著提升

**损失缩放解决梯度下溢**

FP16 的最小正数约为 $6 \times 10^{-8}$，深度网络中大量梯度值会小于此阈值而变为 0（下溢）。

损失缩放的解决思路：

$$L_{\text{scaled}} = L \times S \quad \Rightarrow \quad \nabla_{\text{scaled}} = \nabla L \times S$$

反向传播完成后：

$$\nabla = \nabla_{\text{scaled}} / S$$

缩放因子 $S$（通常 512~65536）将梯度整体上移到 FP16 的有效表示范围，避免下溢。动态 GradScaler 会自动检测梯度溢出并调整 $S$。

**训练流程**

```
前向传播 (FP16) → 损失计算 (FP16) → 缩放损失 (×S)
→ 反向传播 (FP16) → 梯度缩放 (÷S) → 权重更新 (FP32 主副本)
```

---

### 问题 5：如何防止训练不稳定（梯度爆炸、损失发散）？

**答：通过梯度裁剪、合理初始化、学习率 Warmup 和网络结构设计等多种手段综合防控。**

**1. 梯度裁剪（核心手段）**

按全局范数裁剪：

$$g_{\text{norm}} = \sqrt{\sum_i \|g_i\|^2}, \quad g_i \leftarrow g_i \cdot \min\left(1, \frac{\theta}{g_{\text{norm}}}\right)$$

这保持梯度方向不变，只缩放大小。Transformer 训练推荐 `max_norm=1.0`。

**2. 学习率 Warmup**

随机初始化的参数可能产生极大梯度，Warmup 阶段使用小学习率让参数先稳定下来，再逐步增大更新步长。

**3. 层归一化（LayerNorm）**

$$\hat{x} = \frac{x - \mu}{\sigma + \epsilon}, \quad y = \gamma \hat{x} + \beta$$

LayerNorm 将每层输入归一化，防止激活值分布随深度漂移，稳定梯度传播。

**4. 残差连接**

$$y = x + F(x)$$

残差连接为梯度提供"高速通道"，缓解深层网络中的梯度消失问题。

**5. 监控指标**

| 指标 | 正常范围 | 异常信号 |
|------|---------|---------|
| 损失 | 平滑下降 | 突然激增或 NaN |
| 梯度范数 | 0.1~10 | >100 表示爆炸 |
| 参数更新率 | ~1e-3 | 过大表示步长异常 |

**6. 实践清单**

- 使用 AdamW + Warmup + Cosine 调度
- 设置 `clip_grad_norm_=1.0`
- 选择合适的权重初始化（Xavier/He）
- 使用 LayerNorm 而非 BatchNorm
- 定期保存检查点，出现异常可回滚
